# Kinyarwanda News Topic Classification

Classifying Kinyarwanda news articles into 14 topics (politics, sport, economy, health,
entertainment, history, technology, tourism, culture, fashion, religion, environment,
education, relationship) on the **KINNEWS** benchmark (Niyongabo et al., 2020).

| | |
|---|---|
| **Open in Colab** | `[[TODO: Colab badge URL — needs the GitHub repo to exist first]]` |
| **Live app (Hugging Face Space)** | `[[TODO: Space URL — Phase 7]]` |
| **Model on the Hub** | `[[TODO: Hub model URL — Phase 7]]` |
| **Video walkthrough** | `[[TODO: video link]]` |

### What this notebook does

It compares four families of text classifier on the same data and the same splits:

1. **Sparse linear models** — TF-IDF word n-grams and character n-grams + Logistic Regression.
2. **A recurrent model** — a BiLSTM written from scratch in PyTorch, with and without
   fastText subword embeddings trained on the training split.
3. **Multilingual Transformers** — XLM-R base, fine-tuned.
4. **African-centric Transformers** — AfroXLMR base, fine-tuned.

It then runs two ablations, a full evaluation with confidence intervals and significance
tests, a detailed error analysis, and deploys the winner as a Gradio app.

### How to read this notebook

Every code cell is preceded by a markdown cell saying **what** it does, **why** it is done
that way, and **what alternative was rejected**. Every number reported anywhere in this
project is written by code into `results/` and read back from there — no figure in the
report or README is typed by hand.

---
**Build status:** Section 0 is complete (Phase 1). Sections 1–13 are scaffolded and are
filled in by later build phases; each carries a visible `[[TODO: ...]]` marker until then.

## 0. Setup

### 0.1 Environment detection and dependency install

**What this does.** Works out whether we are on Google Colab or a local machine, and on
Colab installs the exact pinned versions from `requirements.txt` so the notebook behaves
the same in both places.

**Why.** The project is developed locally on CPU but the Transformer runs (E3, E4) need a
GPU, which means Colab. Pinning the same versions in both environments is what makes a
result reproducible rather than "it worked on my machine that day".

**Alternative rejected.** Letting Colab use its own preinstalled `transformers`/`torch`.
Rejected because Colab upgrades those images without warning, so a notebook that ran in
September can silently behave differently in October — exactly the kind of drift that
invalidates a reported number.

In [ ]:
import os
import sys
import platform
import subprocess

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

# Pinned versions. Kept identical to requirements.txt at the repo root.
PINNED = [
    "torch==2.14.0",
    "transformers==4.57.6",
    "datasets==4.8.5",
    "accelerate==1.15.0",
    "gensim==4.4.0",
    "scikit-learn==1.8.0",
    "jupytext==1.19.5",
]

if IN_COLAB:
    print("Running on Google Colab -> installing pinned dependencies (quiet, ~2 min).")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PINNED],
        check=True,
    )
else:
    print("Running locally -> assuming the pinned virtualenv is already active.")

print(f"python   : {platform.python_version()}")
print(f"platform : {platform.platform()}")
print(f"in_colab : {IN_COLAB}")

### 0.2 The `CONFIG` dictionary

**What this does.** Defines every tunable value used anywhere in the project — data
choices, preprocessing rules, model hyperparameters, evaluation settings — in one place.

**Why one dict.** Two reasons. First, it is the single thing that has to be copied into a
results file for a run to be reproducible, so every `results/runs/*.json` embeds a copy of
it. Second, it means a hyperparameter can never disagree between the place it is set and
the place it is reported; there is only one place.

**Alternative rejected.** Scattering hyperparameters at the point of use (e.g. writing
`lr=2e-5` inside the training call). That reads more naturally line by line, but it makes
the notebook impossible to audit — you cannot answer "what learning rate produced this
number?" without re-reading every cell.

**On `fast_mode`.** When `True`, the data is subsampled and training is cut to a couple of
steps so the whole notebook runs end to end in under ~10 minutes on a CPU. This is a
smoke test that the code path works — **its numbers are meaningless and are never
reported**. Result files written in fast mode are tagged so they can never be mistaken for
real ones.

In [ ]:
CONFIG = {
    # ---------- identity ----------
    "project": "kinyarwanda-news-classification",
    "seed": 42,
    # fast_mode: subsample + tiny training so the notebook smoke-tests quickly on CPU.
    # Set to False for every number that is actually reported.
    "fast_mode": True,

    # ---------- data ----------
    "data": {
        "hf_id": "kinnews_kirnews",
        # The raw (not pre-cleaned) version: Transformers expect natural text, and this
        # project documents its own preprocessing rather than inheriting someone else's.
        "hf_config": "kinnews_raw",
        "num_labels": 14,
        "val_size": 0.10,          # stratified slice of the official train split
        "label_column": "label",
        "title_column": "title",
        "content_column": "content",
    },

    # ---------- quality audit ----------
    "audit": {
        "near_dup_threshold": 0.90,   # TF-IDF cosine above this = near-duplicate
        "min_words": 10,              # articles shorter than this are flagged
        "label_spot_check_n": 20,     # articles shown for manual review by the student
    },

    # ---------- preprocessing ----------
    "preprocess": {
        # "title_content" = title + " . " + content (default), "title_only" = ablation A1
        "text_mode": "title_content",
        "url_token": "<url>",
        "email_token": "<email>",
        "num_token": "<num>",
    },

    # ---------- B1 / E1: sparse linear models ----------
    "classical": {
        "word_ngram_range": (1, 2),
        "word_min_df": 2,
        "word_max_features": 200_000,
        "char_ngram_range": (2, 5),
        "char_analyzer": "char_wb",
        "char_min_df": 3,
        "char_max_features": 300_000,
        "sublinear_tf": True,
        "C_grid": [0.1, 1.0, 10.0],   # tuned on validation only
        "class_weight": "balanced",
        "max_iter": 2000,
        "top_features_per_class": 15,
    },

    # ---------- E2: BiLSTM ----------
    "bilstm": {
        "embed_dim": 300,
        "hidden_dim": 128,            # per direction; concatenated -> 256
        "num_layers": 1,
        "bidirectional": True,
        "dropout": 0.3,
        "max_tokens": 400,
        "min_freq": 2,                # vocabulary built on the train split only
        "lr": 1e-3,
        "batch_size": 64,
        "epochs": 15,
        "patience": 3,                # early stopping on validation macro-F1
        "grad_clip": 1.0,
        "class_weighted_loss": True,
    },

    # ---------- E2b: fastText embeddings ----------
    "fasttext": {
        "sg": 1,                      # skip-gram (Bojanowski et al., 2017)
        "vector_size": 300,
        "window": 5,
        "min_count": 2,
        "epochs": 10,
        "negative": 5,
        "min_n": 3,                   # character n-gram range for subword vectors
        "max_n": 6,
        "workers": 4,
        "neighbour_probe_words": [],  # filled in Phase 4 with real Kinyarwanda words
    },

    # ---------- E3 / E4: Transformers ----------
    "transformer": {
        "models": {
            "E3": "xlm-roberta-base",
            "E4": "Davlan/afro-xlmr-base",
        },
        "lr": 2e-5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "epochs": 3,
        "max_length": 512,
        "per_device_batch_size": 8,
        "grad_accum_steps": 2,        # effective batch 16
        "fp16": True,                 # GPU only; disabled automatically on CPU
        "metric_for_best_model": "eval_macro_f1",
    },

    # ---------- ablations ----------
    "ablations": {
        "A1_text_mode": "title_only",
        "A2_max_length": 256,
    },

    # ---------- evaluation ----------
    "eval": {
        "primary_metric": "macro_f1",
        "bootstrap_n": 1000,
        "bootstrap_ci": 0.95,
        "n_confident_mistakes": 10,
        "n_taxonomy_errors": 30,
        "length_buckets": [0, 100, 250, 500, 10**9],
    },
}

# fast_mode overrides: applied here so the rest of the notebook never branches on the flag.
if CONFIG["fast_mode"]:
    CONFIG["fast"] = {
        "n_train": 800,
        "n_val": 200,
        "n_test": 200,
    }
    CONFIG["classical"]["C_grid"] = [1.0]
    CONFIG["bilstm"]["epochs"] = 2
    CONFIG["bilstm"]["max_tokens"] = 128
    CONFIG["fasttext"]["epochs"] = 2
    CONFIG["transformer"]["epochs"] = 1
    CONFIG["transformer"]["max_length"] = 128
    CONFIG["eval"]["bootstrap_n"] = 100
    CONFIG["ablations"]["A2_max_length"] = 64

print("fast_mode:", CONFIG["fast_mode"])
if CONFIG["fast_mode"]:
    print("  !! Numbers produced in this mode are a smoke test only and are never reported.")
print(f"seed: {CONFIG['seed']}   primary metric: {CONFIG['eval']['primary_metric']}")

### 0.3 Project paths (and Google Drive on Colab)

**What this does.** Finds the repository root, creates the output folders, and on Colab
mounts Google Drive so checkpoints and result files survive a disconnected session.

**Why Drive.** Colab reclaims its VM without warning. A fine-tuning run that takes 40
minutes and writes only to `/content` is lost when that happens. Writing results to Drive,
plus the "skip if the result JSON already exists" rule used by every experiment, means a
dropped session resumes instead of restarting.

**Alternative rejected.** Re-downloading and re-running everything each session. Rejected
on time: the full experiment suite is several GPU-hours, and Colab sessions do not
reliably last that long.

In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    """Walk up from the current directory looking for a repo marker."""
    markers = ("README.md", "requirements.txt", ".git")
    for start in (Path.cwd(), Path.cwd().parent):
        for candidate in [start, *start.parents]:
            if any((candidate / m).exists() for m in markers):
                return candidate
    return Path.cwd()


DRIVE_ROOT = None
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path("/content/drive/MyDrive") / CONFIG["project"]
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        PROJECT_ROOT = DRIVE_ROOT
        print(f"Drive mounted. Artefacts persist at: {DRIVE_ROOT}")
    except Exception as exc:  # not fatal: the notebook still runs, it just will not resume
        print(f"Drive mount failed ({exc}); falling back to ephemeral /content storage.")
        PROJECT_ROOT = Path("/content") / CONFIG["project"]
else:
    PROJECT_ROOT = find_project_root()

PATHS = {
    "root": PROJECT_ROOT,
    "data": PROJECT_ROOT / "data",
    "audit": PROJECT_ROOT / "data" / "audit",
    "results": PROJECT_ROOT / "results",
    "runs": PROJECT_ROOT / "results" / "runs",
    "predictions": PROJECT_ROOT / "results" / "predictions",
    "figures": PROJECT_ROOT / "figures",
    "report": PROJECT_ROOT / "report_materials",
    "models": PROJECT_ROOT / "models",
    "hf_cache": PROJECT_ROOT / "data" / "hf_cache",
}
for _name, _path in PATHS.items():
    _path.mkdir(parents=True, exist_ok=True)

CONFIG["paths"] = {k: str(v) for k, v in PATHS.items()}

print(f"project root: {PATHS['root']}")
for _name in ("data", "results", "figures", "report", "models"):
    print(f"  {_name:10} -> {PATHS[_name]}")

### 0.4 Seeding

**What this does.** Fixes the random seed for Python, NumPy and PyTorch, and asks cuDNN
for deterministic kernels.

**Why.** Shuffling, the validation split, dropout masks and weight initialisation are all
random. Without a fixed seed, re-running the notebook gives a slightly different macro-F1
and there is no way to tell a real improvement from noise.

**Honest caveat, which belongs in the defense.** This makes a run *repeatable*, not
*robust*. A single seed still does not tell us whether E4 beating E3 by a point is a real
effect or a lucky initialisation — multi-seed training is explicitly out of scope here
(Section 11), which is why the bootstrap confidence intervals and McNemar's test in
Section 8 carry that argument instead.

**Alternative rejected.** Averaging over 3–5 seeds per model. That is the statistically
right answer, and it was rejected purely on compute budget: it would multiply the
Transformer runs, which dominate the cost, by 3–5.

In [ ]:
import random

import numpy as np


def set_seed(seed: int) -> None:
    """Seed every source of randomness the project touches."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


set_seed(CONFIG["seed"])
print(f"seeded everything with {CONFIG['seed']}")

### 0.5 Hardware check

**What this does.** Reports the device the notebook will train on, and turns off `fp16`
when there is no GPU.

**Why it is printed.** Training time and the feasibility of E3/E4 depend entirely on this.
`fp16` (half precision) is a GPU feature; leaving it on for a CPU run raises an error, so
the flag is derived from the hardware rather than trusted from `CONFIG`.

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG["transformer"]["fp16"] = bool(torch.cuda.is_available()) and CONFIG["transformer"]["fp16"]

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu          : {_props.name} ({_props.total_memory / 1e9:.1f} GB)")
else:
    print(f"cpu threads  : {torch.get_num_threads()}")
    print("  No GPU: E3/E4 full fine-tuning is not practical here. Use fast_mode locally")
    print("  for a smoke test and run the real Transformer experiments on Colab.")
print(f"fp16 enabled : {CONFIG['transformer']['fp16']}")

## 1. Introduction and research questions

**The task.** Given the text of a Kinyarwanda news article, predict which of 14 topics it
belongs to: politics, sport, economy, health, entertainment, history, technology, tourism,
culture, fashion, religion, environment, education, relationship.

**Why Kinyarwanda.** Kinyarwanda is the national language of Rwanda, spoken by roughly 12
million people, yet it sits in the long tail of NLP resource availability described by
Joshi et al. (2020): many speakers, very little labelled data or tooling. It is also
**agglutinative** and has a **noun-class system**, so a single stem surfaces in many
different written forms depending on its prefixes and affixes. That property is not a
footnote here — it is the reason RQ1 and RQ2 are worth asking, because both character
n-grams and fastText subword vectors are mechanisms for sharing evidence across the
surface forms of one stem.

**Who would use this.** Newsroom and archive staff tagging incoming articles, aggregators
routing content by topic, and researchers or developers who want a documented Kinyarwanda
classifier rather than building one from nothing.

### The four research questions

Every experiment in this notebook exists to answer one of these, and every
"What we learned" cell points back to them.

| RQ | Question | Why it matters for Kinyarwanda |
|---|---|---|
| **RQ1** | How far can a simple sparse model go, and do character n-grams beat word n-grams? | Agglutination means word-level features fragment one stem across many types. Character n-grams share signal across those forms. |
| **RQ2** | Do subword embeddings trained on in-domain text help a BiLSTM over random initialisation? | fastText composes word vectors from character n-grams, so it can represent forms never seen in training. |
| **RQ3** | Does African-centric pretraining (AfroXLMR) beat general multilingual pretraining (XLM-R)? | Tests directly whether adapting a large model to African languages pays off on a real downstream task. |
| **RQ4** | What does the best model still get wrong, and why? | Moves the project from reporting a score to understanding the model. |

**Out of scope, deliberately:** cross-lingual transfer to Kirundi (KIRNEWS), LoRA and other
parameter-efficient methods, multi-seed training, probability calibration, and sentiment.
These are revisited in Section 11 as future work.

## 2. Related work notes

`[[TODO: Phase 8 — short, cited; mirrors report_materials/related_work_notes.md]]`

## 3. Dataset

### 3.1 Loading KINNEWS

**What this does.** Tries the four loading routes from the project specification in order
and records which one actually worked, then caches the result to `data/`.

**Why four routes.** KINNEWS is an older Hub dataset that ships as a *loading script*
rather than as data files. `datasets` removed the ability to execute such scripts in
version 3.0, so the obvious one-liner is no longer guaranteed to work. Rather than assume,
the notebook attempts each route and prints the outcome, so the loading path is a recorded
fact rather than a claim.

**Alternative rejected.** Hard-coding the route that works today. Rejected because it hides
*why* the others fail, which is exactly the kind of question an examiner asks — and because
the Hub's behaviour for script datasets may change again.

In [ ]:
import hashlib
import io
import json
import re
import time
import unicodedata
import urllib.request
import zipfile

import pandas as pd

RAW_DIR = PATHS["data"] / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# The loader script on the Hub points at this mirror of the authors' release.
KINNEWS_ZIP_URL = "https://github.com/saradhix/kinnews_kirnews/raw/master/KINNEWS.zip"
KINNEWS_HOMEPAGE = "https://github.com/Andrews2017/KINNEWS-and-KIRNEWS-Corpus"
LOAD_LOG = []


def _record(route, status, detail):
    LOAD_LOG.append({"route": route, "status": status, "detail": str(detail)[:200]})
    print(f"  [{status:8}] route {route}: {str(detail)[:150]}")


def _route_1_hub_script():
    from datasets import load_dataset

    return load_dataset(CONFIG["data"]["hf_id"], CONFIG["data"]["hf_config"])


def _route_2_parquet():
    from datasets import load_dataset

    return load_dataset(
        CONFIG["data"]["hf_id"], CONFIG["data"]["hf_config"], revision="refs/convert/parquet"
    )


def _route_4_authors_release():
    """Download the authors' archive and read the raw CSVs directly."""
    zip_path = RAW_DIR / "KINNEWS.zip"
    if not zip_path.exists():
        req = urllib.request.Request(KINNEWS_ZIP_URL, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=300) as resp, open(zip_path, "wb") as fh:
            fh.write(resp.read())
    digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    print(f"  archive: {zip_path.stat().st_size:,} bytes  sha256={digest[:16]}...")
    frames = {}
    with zipfile.ZipFile(zip_path) as zf:
        for split in ("train", "test"):
            with zf.open(f"KINNEWS/raw/{split}.csv") as fh:
                frames[split] = pd.read_csv(io.BytesIO(fh.read()), encoding="utf-8")
        class_names = zf.read("KINNEWS/classes.txt").decode("utf-8").split()
    return frames, class_names, digest


print("Attempting the four loading routes in order:\n")

train_df = test_df = class_names = None

try:
    _ds = _route_1_hub_script()
    _record(1, "OK", "load_dataset with the Hub loading script succeeded")
    train_df, test_df = _ds["train"].to_pandas(), _ds["test"].to_pandas()
except Exception as exc:
    _record(1, "FAILED", f"{type(exc).__name__}: {exc}")

if train_df is None:
    try:
        _ds = _route_2_parquet()
        _record(2, "OK", "Hub parquet conversion succeeded")
        train_df, test_df = _ds["train"].to_pandas(), _ds["test"].to_pandas()
    except Exception as exc:
        _record(2, "FAILED", f"{type(exc).__name__}: {exc}")

# Route 3 (pin datasets<3.0 + trust_remote_code) cannot be attempted from inside a running
# kernel: it needs a different, conflicting dependency set. It is documented in
# DECISIONS.md (D4) and in requirements.txt as the fallback if route 4 ever breaks.
_record(3, "SKIPPED", "would require downgrading `datasets` in-session; see DECISIONS.md D4")

if train_df is None:
    _frames, class_names, _digest = _route_4_authors_release()
    train_df, test_df = _frames["train"], _frames["test"]
    _record(4, "OK", f"authors' release parsed directly ({KINNEWS_HOMEPAGE})")

if class_names is None:
    class_names = [
        "politics", "sport", "economy", "health", "entertainment", "history", "technology",
        "tourism", "culture", "fashion", "religion", "environment", "education", "relationship",
    ]

LABELS = list(class_names)
CONFIG["data"]["class_names"] = LABELS
print(f"\nLoaded: train={train_df.shape}, test={test_df.shape}")
print(f"Classes ({len(LABELS)}): {', '.join(LABELS)}")

### 3.2 Confirming the published description

**What this does.** Checks the dataset against what the KINNEWS paper and the archive's own
`readme.txt` claim — row counts, number of classes, field names, label range, missing
values — and prints an explicit PASS/FAIL for each.

**Why bother, when the paper already says so.** Because a published description and the
file you actually downloaded are two different things, and the whole project rests on this
file. Checking takes seconds and converts an assumption into evidence. Where the data
disagrees with the description, the notebook reports the data.

In [ ]:
CLAIMS = {
    "train rows == 17,014": (len(train_df), 17_014),
    "test rows == 4,254": (len(test_df), 4_254),
    "classes == 14": (train_df["label"].nunique(), 14),
    "labels are 1..14": ((train_df["label"].min(), train_df["label"].max()), (1, 14)),
}
print("Published claim vs. the file on disk:\n")
for claim, (actual, expected) in CLAIMS.items():
    print(f"  {'PASS' if actual == expected else 'FAIL'}  {claim:24} actual={actual}")

expected_cols = ["label", "kin_label", "en_label", "url", "title", "content"]
print(f"\n  {'PASS' if list(train_df.columns) == expected_cols else 'FAIL'}  "
      f"columns == {expected_cols}")
print(f"\nMissing values (train): {train_df.isna().sum().to_dict()}")
print(f"Missing values (test) : {test_df.isna().sum().to_dict()}")

# Label index -> names, taken from the data itself rather than assumed.
label_map = (
    train_df.groupby("label")[["kin_label", "en_label"]]
    .agg(lambda s: s.mode().iat[0])
    .assign(train_n=train_df["label"].value_counts().sort_index(),
            test_n=test_df["label"].value_counts().sort_index())
)
label_map["train_%"] = (label_map.train_n / len(train_df) * 100).round(2)
label_map["test_%"] = (label_map.test_n / len(test_df) * 100).round(2)
print("\nClass index -> Kinyarwanda / English label, with counts:\n")
print(label_map.to_string())

_imb = label_map.train_n.max() / label_map.train_n.min()
print(f"\nTrain imbalance ratio (largest/smallest class): {_imb:.1f}x")
print(f"  largest : {label_map.train_n.idxmax()} = {label_map.loc[label_map.train_n.idxmax(),'en_label']}"
      f" ({label_map.train_n.max():,})")
print(f"  smallest: {label_map.train_n.idxmin()} = {label_map.loc[label_map.train_n.idxmin(),'en_label']}"
      f" ({label_map.train_n.min():,})")

**Two corrections to the project brief, both established above from the data.**

1. The brief states that *"politics is the largest class and education the smallest."*
   Politics is indeed the largest training class, but the **smallest is environment**
   (38 articles), followed by fashion (54); education has 116. Reported as found.

2. Far more consequential: the official **train and test splits do not share a class
   distribution.** Environment is ~19x more prevalent in test than in train, fashion ~11x,
   while economy is ~0.2x. This is a property of the published benchmark, not of anything
   done here, and it shapes how every later result must be read — a model that learns
   training priors well will be actively penalised on this test set. It is quantified in
   Section 3.5 and revisited in the error analysis.

### 3.3 Quality audit

**What this does.** Looks for exact duplicates, near-duplicates, train/test leakage,
degenerate short articles, and label noise — then decides what to remove.

**Why it runs *before* the split.** The specification lists the split before the audit, but
the order is reversed here for a concrete reason: if an article appears three times in the
training file and the validation split is taken afterwards, copies of the same article land
on both sides of the split. Validation macro-F1 then measures memorisation and every model
selection decision in the project is made on a corrupted signal. Deduplicating first makes
the validation split honest.

**Alternative rejected.** Splitting first and deduplicating within each side afterwards.
That removes the leakage too, but it makes the split sizes depend on duplicate structure
and is harder to explain. Deduplicating first is both safer and simpler to defend.

In [ ]:
def normalise_for_dedup(text: str) -> str:
    """Aggressive normalisation used ONLY to detect duplicates, never for modelling."""
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


for _df in (train_df, test_df):
    _df["full_text"] = _df["title"].fillna("") + " " + _df["content"].fillna("")
    _df["dedup_key"] = _df["full_text"].map(normalise_for_dedup)

audit = {}
audit["train_rows"] = len(train_df)
audit["test_rows"] = len(test_df)
audit["train_exact_dup_rows_raw"] = int(train_df["full_text"].duplicated().sum())
audit["train_exact_dup_rows_normalised"] = int(train_df["dedup_key"].duplicated().sum())
audit["train_duplicate_url_rows"] = int(train_df["url"].duplicated().sum())
audit["train_distinct_texts"] = int(train_df["dedup_key"].nunique())
audit["test_exact_dup_rows_normalised"] = int(test_df["dedup_key"].duplicated().sum())

print("Exact duplicates")
print(f"  train rows that repeat earlier text (untouched) : {audit['train_exact_dup_rows_raw']:,}")
print(f"  train rows that repeat earlier text (normalised): {audit['train_exact_dup_rows_normalised']:,}")
print(f"  train rows with a repeated URL                  : {audit['train_duplicate_url_rows']:,}")
print(f"  distinct training texts                         : {audit['train_distinct_texts']:,}"
      f" of {len(train_df):,}"
      f"  ({100*audit['train_exact_dup_rows_normalised']/len(train_df):.1f}% are copies)")
print(f"  test rows that repeat earlier text (normalised) : {audit['test_exact_dup_rows_normalised']:,}")

_grp = train_df.groupby("dedup_key")
_rep = _grp.size().sort_values(ascending=False)
print(f"\n  most-repeated single article appears {_rep.iloc[0]}x")
print(f"  title: {train_df.loc[train_df.dedup_key == _rep.index[0], 'title'].iloc[0][:100]}")

# Duplicate groups that disagree about the label are irreducible label noise.
_nlab = _grp["label"].nunique()
_conflict = _nlab[_nlab > 1]
audit["dup_groups_with_conflicting_labels"] = int(len(_conflict))
audit["rows_in_label_conflicting_groups"] = int(train_df["dedup_key"].isin(_conflict.index).sum())
print(f"\nLabel noise revealed by duplicates")
print(f"  duplicate groups carrying >1 distinct label: {audit['dup_groups_with_conflicting_labels']:,}")
print(f"  training rows involved                     : {audit['rows_in_label_conflicting_groups']:,}")
print("  -> identical text labelled two different ways puts a ceiling on achievable accuracy.")

**Leakage check.** The question that actually matters: does any training article also
appear in the official test set? Exact overlap is checked first, then near-duplicates by
TF-IDF cosine similarity, which catches lightly edited republications that exact matching
misses.

The similarity matrix is computed in chunks because the full train x test matrix would not
fit comfortably in memory on a modest machine.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

test_keys = set(test_df["dedup_key"])
leak_mask = train_df["dedup_key"].isin(test_keys)
audit["train_test_exact_overlap_rows"] = int(leak_mask.sum())
print(f"Exact train/test overlap: {audit['train_test_exact_overlap_rows']} training rows "
      f"are verbatim copies of a test article.")

_t0 = time.time()
_dedup_train = train_df.drop_duplicates("dedup_key", keep="first")
_vec = TfidfVectorizer(min_df=2, max_features=60_000, ngram_range=(1, 1))
_Xtr = normalize(_vec.fit_transform(_dedup_train["dedup_key"]))
_Xte = normalize(_vec.transform(test_df["dedup_key"]))

_near, _best = 0, []
_thr = CONFIG["audit"]["near_dup_threshold"]
for _s in range(0, _Xte.shape[0], 500):
    _sim = (_Xte[_s:_s + 500] @ _Xtr.T).toarray()
    _mx = _sim.max(axis=1)
    _near += int((_mx > _thr).sum())
    _best.append(_mx)
_best = np.concatenate(_best)
audit["train_test_near_dup_rows"] = _near
audit["near_dup_threshold"] = _thr
print(f"Near-duplicates (TF-IDF cosine > {_thr}): {_near} of {len(test_df):,} test articles "
      f"have a very similar training article  ({100*_near/len(test_df):.2f}%)")
print(f"  max similarity seen: {_best.max():.3f};  median: {np.median(_best):.3f}"
      f"   [computed in {time.time()-_t0:.1f}s]")

**Short and empty articles.** Counted and inspected rather than silently dropped.

In [ ]:
train_df["n_words"] = train_df["full_text"].str.split().str.len()
test_df["n_words"] = test_df["full_text"].str.split().str.len()
_min_w = CONFIG["audit"]["min_words"]
audit["train_short_articles"] = int((train_df.n_words < _min_w).sum())
audit["test_short_articles"] = int((test_df.n_words < _min_w).sum())
audit["test_null_content"] = int(test_df["content"].isna().sum())
print(f"Articles under {_min_w} words:  train={audit['train_short_articles']}, "
      f"test={audit['test_short_articles']}")
print(f"Test rows with empty content (title only): {audit['test_null_content']}")
if audit["test_null_content"]:
    print("  -> kept: they still carry a title, and the test set is never modified.")
print(f"Shortest training article: {train_df.n_words.min()} words")

**Label spot check.** Twenty random articles with title, snippet and assigned label.

This table is printed and saved **for manual review by the student**. The notebook makes no
claim about whether any of these labels is wrong — judging that requires reading
Kinyarwanda, which is the student's job, not the code's.

In [ ]:
_spot = train_df.sample(CONFIG["audit"]["label_spot_check_n"], random_state=CONFIG["seed"])
spot_check = pd.DataFrame({
    "row": _spot.index,
    "label": _spot["en_label"].values,
    "title": _spot["title"].str.slice(0, 70).values,
    "snippet": _spot["content"].astype(str).str.slice(0, 90).values,
})
spot_check.to_csv(PATHS["audit"] / "label_spot_check.csv", index=False)
print("FOR MANUAL REVIEW BY THE STUDENT — the notebook asserts nothing about these labels.\n")
print(spot_check.to_string(index=False))
print(f"\nsaved -> {PATHS['audit'] / 'label_spot_check.csv'}")

**The decision, and its effect.** Two removals, both applied to training data only:

1. Drop exact duplicates within train, keeping the first occurrence.
2. Drop any training article that is a verbatim copy of a test article.

**The test set is not touched at all** — not deduplicated, not filtered. It stays exactly
as published so results remain comparable with the KINNEWS paper and with anyone else who
uses this benchmark.

Dropping 55% of the training rows sounds drastic, so it is worth being precise about what
it costs: it removes **no information**, because every removed row is a byte-identical copy
of a row that is kept. What it removes is the *illusion* of a larger dataset, and the
leakage between train and validation that would otherwise follow.

In [ ]:
_before = len(train_df)
train_df = train_df[~leak_mask].drop_duplicates("dedup_key", keep="first").reset_index(drop=True)
audit["train_rows_after_dedup"] = len(train_df)
audit["train_rows_removed"] = _before - len(train_df)

_counts_after = train_df["label"].value_counts().sort_index()
audit["imbalance_ratio_before"] = round(float(_imb), 1)
audit["imbalance_ratio_after"] = round(float(_counts_after.max() / _counts_after.min()), 1)

print(f"train: {_before:,} -> {len(train_df):,} rows "
      f"({audit['train_rows_removed']:,} removed, {100*audit['train_rows_removed']/_before:.1f}%)")
print(f"test : {len(test_df):,} rows (untouched, as published)")
print(f"\nimbalance ratio: {audit['imbalance_ratio_before']}x -> "
      f"{audit['imbalance_ratio_after']}x after deduplication")
print("\nDuplicates removed per class (they were concentrated in the biggest classes):")
_removed = (label_map.train_n - _counts_after).fillna(0).astype(int)
print(pd.DataFrame({"class": label_map.en_label, "removed": _removed,
                    "remaining": _counts_after}).to_string())

with open(PATHS["audit"] / "audit_summary.json", "w") as fh:
    json.dump(audit, fh, indent=2)
print(f"\nsaved -> {PATHS['audit'] / 'audit_summary.json'}")

### 3.4 Splits

**What this does.** Carves a stratified 10% validation set out of the deduplicated training
data and writes every index to `data/splits.json`, so that every model in this notebook
trains and is selected on byte-identical data.

**Why stratified.** With environment down to 38 articles, a random 10% slice could easily
leave that class with almost nothing in validation, making its validation F1 pure noise —
and macro-F1 averages over classes, so one noisy class moves the number the whole project
is selected on.

**Why persist the indices.** So that "E4 beat B1" is a statement about models rather than
about two different shuffles. It also lets a Colab session that died mid-run resume on the
same data.

**Alternative rejected.** Cross-validation, which would use the data better on a set this
small. Rejected because it multiplies every Transformer fine-tune by k, and the GPU budget
here does not allow that.

In [ ]:
from sklearn.model_selection import train_test_split

_tr_idx, _va_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=CONFIG["data"]["val_size"],
    random_state=CONFIG["seed"],
    stratify=train_df["label"].values,
)
splits = {
    "seed": CONFIG["seed"],
    "val_size": CONFIG["data"]["val_size"],
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "note": "Indices are positions in the DEDUPLICATED train frame; test is the official split, untouched.",
    "audit": audit,
    "train_idx": sorted(int(i) for i in _tr_idx),
    "val_idx": sorted(int(i) for i in _va_idx),
    "n_test": len(test_df),
}
with open(PATHS["data"] / "splits.json", "w") as fh:
    json.dump(splits, fh, indent=2)

tr_df = train_df.iloc[splits["train_idx"]].reset_index(drop=True)
va_df = train_df.iloc[splits["val_idx"]].reset_index(drop=True)
te_df = test_df.reset_index(drop=True)

print(f"train {len(tr_df):,}   val {len(va_df):,}   test {len(te_df):,}")
print(f"saved -> {PATHS['data'] / 'splits.json'}")

# The leakage guarantee, asserted rather than assumed.
assert set(tr_df.dedup_key) & set(va_df.dedup_key) == set(), "train/val overlap!"
assert set(tr_df.dedup_key) & set(te_df.dedup_key) == set(), "train/test overlap!"
assert set(va_df.dedup_key) & set(te_df.dedup_key) == set(), "val/test overlap!"
print("\nPASS  no article text is shared between train, validation and test.")

split_counts = pd.DataFrame({
    "class": label_map.en_label,
    "train": tr_df.label.value_counts().sort_index(),
    "val": va_df.label.value_counts().sort_index(),
    "test": te_df.label.value_counts().sort_index(),
}).fillna(0).astype({"train": int, "val": int, "test": int})
print("\nClass counts per split:\n")
print(split_counts.to_string())

### 3.5 Exploratory data analysis

**What this does.** Five figures, each saved to `figures/` at 200 dpi. Each one exists to
justify a decision made later in the notebook, not to decorate the report.

| Figure | Justifies |
|---|---|
| Class distribution per split | macro-F1 as the primary metric; class-weighted losses |
| Train vs test distribution shift | how every test number must be interpreted |
| Length distribution (words and subword tokens) | `max_length` 512, and ablation A2 |
| Tokenizer fertility | the Kinyarwanda morphology argument behind RQ1/RQ2/RQ3 |
| Vocabulary growth and hapax rate | why character n-grams and subwords should help |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
FIGDPI = 200


def save_fig(fig, name: str) -> None:
    path = PATHS["figures"] / f"{name}.png"
    fig.savefig(path, dpi=FIGDPI, bbox_inches="tight")
    print(f"saved -> {path}")


# ---- Figure 1: class distribution per split ----
fig, ax = plt.subplots(figsize=(11, 5))
_plot = split_counts.set_index("class")[["train", "val", "test"]]
_plot.plot(kind="bar", ax=ax, width=0.8)
ax.set_title(f"KINNEWS class distribution by split "
             f"(train imbalance {audit['imbalance_ratio_after']}x after deduplication)")
ax.set_xlabel("Topic")
ax.set_ylabel("Number of articles")
ax.legend(title="Split")
plt.xticks(rotation=45, ha="right")
save_fig(fig, "01_class_distribution_by_split")
plt.show()

**Figure 2 is the most important plot in this section.** It compares each class's *share*
of the training set against its share of the test set. A value of 1.0 means the class is
equally represented in both; the log scale is necessary because the real values span two
orders of magnitude.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
_share = pd.DataFrame({
    "train_%": tr_df.label.value_counts(normalize=True).sort_index() * 100,
    "test_%": te_df.label.value_counts(normalize=True).sort_index() * 100,
}).fillna(0)
_share.index = label_map.en_label.values
_share["ratio"] = _share["test_%"] / _share["train_%"].replace(0, np.nan)
_colors = ["#c0392b" if r > 2 or r < 0.5 else "#7f8c8d" for r in _share["ratio"]]
ax.bar(_share.index, _share["ratio"], color=_colors)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="equal representation")
ax.set_yscale("log")
ax.set_title("Distribution shift: how much more common each topic is in test than in train")
ax.set_xlabel("Topic")
ax.set_ylabel("test share / train share  (log scale)")
ax.legend()
plt.xticks(rotation=45, ha="right")
save_fig(fig, "02_train_test_distribution_shift")
plt.show()
print(_share.round(2).to_string())

**Length.** Measured in whitespace words and in XLM-R subword tokens. The subword count is
what actually decides `max_length`, and the share of articles exceeding 256 and 512 tokens
is exactly the cost that ablation A2 measures.

In [ ]:
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging

# We tokenise with truncation=False on purpose -- measuring true length is the whole point --
# so the "sequence longer than model maximum" notice is expected and would only be noise.
hf_logging.set_verbosity_error()
os.environ.setdefault("HF_HOME", str(PATHS["hf_cache"]))
tok_xlmr = AutoTokenizer.from_pretrained(CONFIG["transformer"]["models"]["E3"])

_sample = tr_df["full_text"].sample(min(1500, len(tr_df)), random_state=CONFIG["seed"])
_tok_lens = np.array([len(tok_xlmr(t, truncation=False)["input_ids"]) for t in _sample])
_word_lens = tr_df["n_words"].values

over_256 = float((_tok_lens > 256).mean() * 100)
over_512 = float((_tok_lens > 512).mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.clip(_word_lens, 0, 1500), bins=60, color="#2980b9")
axes[0].axvline(np.median(_word_lens), color="black", linestyle="--",
                label=f"median {np.median(_word_lens):.0f}")
axes[0].set_title("Article length in words (train)")
axes[0].set_xlabel("Words (clipped at 1500)")
axes[0].set_ylabel("Articles")
axes[0].legend()

axes[1].hist(np.clip(_tok_lens, 0, 2000), bins=60, color="#27ae60")
for _cut, _c in ((256, "#e67e22"), (512, "#c0392b")):
    axes[1].axvline(_cut, color=_c, linestyle="--", label=f"max_length {_cut}")
axes[1].set_title("Article length in XLM-R subword tokens (sample)")
axes[1].set_xlabel("Subword tokens (clipped at 2000)")
axes[1].set_ylabel("Articles")
axes[1].legend()
save_fig(fig, "03_length_distributions")
plt.show()

print(f"words  : mean {_word_lens.mean():.0f}, median {np.median(_word_lens):.0f}, "
      f"max {_word_lens.max():,}")
print(f"tokens : mean {_tok_lens.mean():.0f}, median {np.median(_tok_lens):.0f}, "
      f"max {_tok_lens.max():,}")
print(f"\n{over_256:.1f}% of articles exceed 256 subword tokens")
print(f"{over_512:.1f}% of articles exceed 512 subword tokens")
print(f"-> at max_length=512 the model still never sees the tail of {over_512:.1f}% of articles.")

**Tokenizer fertility** is the average number of subword tokens a tokenizer needs per
whitespace word. High fertility means the language is being shredded into small pieces:
the model spends its limited context on fragments, and each piece carries less meaning.
It is a standard diagnostic for how well a multilingual tokenizer serves a given language.

This is measured for both XLM-R and AfroXLMR because the project brief expects fertility to
help explain any RQ3 difference between them. The result below is worth reading carefully.

In [ ]:
tok_afro = AutoTokenizer.from_pretrained(CONFIG["transformer"]["models"]["E4"])

_fert_sample = tr_df["full_text"].sample(min(400, len(tr_df)), random_state=CONFIG["seed"]).tolist()
fertility = {}
for _name, _tk in (("XLM-R", tok_xlmr), ("AfroXLMR", tok_afro)):
    _n_tok = sum(len(_tk.tokenize(t)) for t in _fert_sample)
    _n_wrd = sum(len(t.split()) for t in _fert_sample)
    fertility[_name] = _n_tok / _n_wrd

identical = (
    tok_xlmr.vocab_size == tok_afro.vocab_size
    and all(tok_xlmr.tokenize(t) == tok_afro.tokenize(t) for t in _fert_sample[:50])
)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(list(fertility), list(fertility.values()), color=["#8e44ad", "#16a085"])
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="1 token per word (ideal)")
for i, (k, v) in enumerate(fertility.items()):
    ax.text(i, v + 0.03, f"{v:.2f}", ha="center", fontweight="bold")
ax.set_title("Tokenizer fertility on Kinyarwanda news\n(subword tokens per whitespace word)")
ax.set_ylabel("Tokens per word")
ax.legend()
save_fig(fig, "04_tokenizer_fertility")
plt.show()

for _k, _v in fertility.items():
    print(f"{_k:10}: {_v:.3f} subword tokens per word  (vocab {tok_xlmr.vocab_size:,})")
print(f"\nDo the two tokenizers produce identical output? {identical}")
print("Example word split by XLM-R:", tok_xlmr.tokenize("Umuryango w'abibumbye uratangaza"))

**This is a genuinely important result for RQ3, and it corrects an assumption in the
project brief.** AfroXLMR reuses XLM-R's tokenizer unchanged — same 250k vocabulary, same
output on every string tested. Fertility is therefore *identical* for the two models and
**cannot explain any difference between them**. Whatever gap Section 6 finds between E3 and
E4 must come from the adapted *weights* (multilingual adaptive fine-tuning), not from
better tokenization. That is a sharper and more defensible version of RQ3 than the one the
brief anticipated.

The absolute fertility figure still matters: at well above one token per word, Kinyarwanda
is being split into many pieces per word, which is the concrete form of the "under-served
by multilingual tokenizers" problem and motivates RQ1 and RQ2.

In [ ]:
# ---- Figure 5: vocabulary richness ----
_tokens = " ".join(tr_df["full_text"].str.lower()).split()
_vocab = pd.Series(_tokens).value_counts()
hapax = float((_vocab == 1).mean() * 100)
ttr = len(_vocab) / len(_tokens)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
_cum = np.cumsum(_vocab.values) / len(_tokens) * 100
axes[0].plot(np.arange(1, len(_cum) + 1), _cum, color="#c0392b")
axes[0].set_xscale("log")
axes[0].set_title("Corpus coverage by vocabulary size")
axes[0].set_xlabel("Distinct word types (log scale)")
axes[0].set_ylabel("% of all word tokens covered")
axes[0].axhline(90, linestyle="--", color="grey", linewidth=1)

axes[1].bar(["appears once\n(hapax)", "appears 2-5x", "appears >5x"],
            [(_vocab == 1).sum(), ((_vocab >= 2) & (_vocab <= 5)).sum(), (_vocab > 5).sum()],
            color=["#c0392b", "#e67e22", "#27ae60"])
axes[1].set_title(f"Word types by frequency  ({hapax:.0f}% appear exactly once)")
axes[1].set_ylabel("Number of word types")
save_fig(fig, "05_vocabulary_richness")
plt.show()

print(f"word tokens      : {len(_tokens):,}")
print(f"distinct types   : {len(_vocab):,}")
print(f"type/token ratio : {ttr:.4f}")
print(f"hapax legomena   : {hapax:.1f}% of types appear exactly once")
print("-> a word-level model must treat most of its vocabulary as near-unseen. This is the")
print("   agglutination argument behind RQ1 (character n-grams) and RQ2 (fastText subwords).")

**Top terms per class**, as a sanity check that the labels mean what they claim. If the
most distinctive words for `sport` are football words, the labels are trustworthy enough
to model.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

_cv = CountVectorizer(min_df=3, max_features=40_000)
_X = _cv.fit_transform(tr_df["full_text"].str.lower())
_terms = np.array(_cv.get_feature_names_out())
_rows = []
for _lab in sorted(tr_df.label.unique()):
    _m = (tr_df.label == _lab).values
    _in = np.asarray(_X[_m].sum(0)).ravel() / max(_m.sum(), 1)
    _out = np.asarray(_X[~_m].sum(0)).ravel() / max((~_m).sum(), 1)
    _score = (_in + 0.5) / (_out + 0.5)
    _rows.append({"class": label_map.loc[_lab, "en_label"],
                  "distinctive terms": ", ".join(_terms[np.argsort(-_score)[:8]])})
print(pd.DataFrame(_rows).to_string(index=False))

### 3.6 Preprocessing

**Two functions, because the two model families want opposite things.**

`preprocess_classical` is aggressive: NFC normalisation, lowercasing, and URLs, emails and
numbers collapsed to placeholder tokens. Sparse models and the BiLSTM build their own
vocabulary from this corpus alone, so every distinct surface form costs a feature or an
embedding row. Collapsing "2019", "2020" and "45" into one `<num>` token spends those
parameters on something useful instead. **Apostrophes are kept**: in Kinyarwanda
orthography they mark elision (`w'abibumbye`, `n'ibindi`) and are part of the word, not
punctuation.

`preprocess_transformer` does almost nothing — NFC and whitespace tidying only. The
tokenizer was trained on natural text, so lowercasing or stripping numbers moves the input
*away* from the distribution the model saw during pretraining and destroys the casing cues
it learned to use. Cleaning here would actively hurt.

**Alternative rejected.** One shared preprocessing function for all models. Simpler, and
wrong in one direction or the other: it would either starve the sparse models of
normalisation or feed the Transformers text unlike anything in their pretraining.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
NUM_RE = re.compile(r"\b\d[\d.,]*\b")
KEEP_RE = re.compile(r"[^\w\s'’<>]", flags=re.UNICODE)


def preprocess_classical(text: str) -> str:
    """For TF-IDF and the BiLSTM: normalise hard, keep apostrophes."""
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = URL_RE.sub(f" {CONFIG['preprocess']['url_token']} ", text)
    text = EMAIL_RE.sub(f" {CONFIG['preprocess']['email_token']} ", text)
    text = NUM_RE.sub(f" {CONFIG['preprocess']['num_token']} ", text)
    text = KEEP_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def preprocess_transformer(text: str) -> str:
    """For XLM-R / AfroXLMR: leave the text looking like natural news prose."""
    text = unicodedata.normalize("NFC", str(text))
    return re.sub(r"\s+", " ", text).strip()


def build_text(df: pd.DataFrame, mode: str = None) -> pd.Series:
    """title + ' . ' + content by default; title only for ablation A1."""
    mode = mode or CONFIG["preprocess"]["text_mode"]
    title = df["title"].fillna("")
    if mode == "title_only":
        return title
    return title + " . " + df["content"].fillna("")


_demo = build_text(tr_df).iloc[0]
print("RAW           :", _demo[:150].replace("\n", " "))
print("\nCLASSICAL     :", preprocess_classical(_demo)[:150])
print("\nTRANSFORMER   :", preprocess_transformer(_demo)[:150])

for _name, _d in (("train", tr_df), ("val", va_df), ("test", te_df)):
    _d["text_classical"] = build_text(_d).map(preprocess_classical)
    _d["text_transformer"] = build_text(_d).map(preprocess_transformer)
    _d["y"] = _d["label"].astype(int) - 1          # 1..14 -> 0..13 for scikit-learn / torch
print(f"\nprepared: train={len(tr_df):,}  val={len(va_df):,}  test={len(te_df):,}")
print(f"label encoding: 1..14 -> 0..13, class_names[0] = {LABELS[0]!r}")


# Fast-mode subsampling happens HERE, at the very end of the data section, and deliberately
# not before it. Everything above describes the dataset -- class balance, lengths, tokenizer
# fertility, vocabulary -- and those are properties of KINNEWS, not of a smoke test. If the
# subsample ran earlier, the figures committed to figures/ would describe a 700-row toy
# sample and the distribution-shift plot in particular would show nothing at all.
if CONFIG["fast_mode"]:
    _per_class = max(2, CONFIG["fast"]["n_train"] // len(LABELS))
    tr_df = pd.concat([g.sample(min(len(g), _per_class), random_state=CONFIG["seed"])
                       for _, g in tr_df.groupby("label")])
    va_df = va_df.sample(min(len(va_df), CONFIG["fast"]["n_val"]), random_state=CONFIG["seed"])
    te_df = te_df.sample(min(len(te_df), CONFIG["fast"]["n_test"]), random_state=CONFIG["seed"])
    tr_df, va_df, te_df = (d.reset_index(drop=True) for d in (tr_df, va_df, te_df))
    print(f"\nFAST MODE: modelling frames subsampled to train={len(tr_df)}, "
          f"val={len(va_df)}, test={len(te_df)} — smoke test only, never reported.")
    print("(The EDA above used the full splits, as it must.)")

## 4. Baselines (B0, B1)

`[[TODO: Phase 3]]`

## 5. Classical and recurrent experiments (E1, E2a, E2b)

`[[TODO: Phase 3 for E1, Phase 4 for E2a/E2b]]`

## 6. Transformer experiments (E3, E4)

`[[TODO: Phase 5]]`

## 7. Ablations (A1, A2)

`[[TODO: Phase 5]]`

## 8. Evaluation of the final model

`[[TODO: Phase 6]]`

## 9. Error analysis

`[[TODO: Phase 6]]`

## 10. Deployment

`[[TODO: Phase 7]]`

## 11. Limitations and future work

`[[TODO: Phase 8]]`

## 12. Conclusion

`[[TODO: Phase 8 — one to three sentences per RQ, with real numbers]]`

## 13. References

`[[TODO: Phase 8 — mirrors report_materials/references.bib, every entry verified]]`